# Data cleaning and standardization

### Summary of the cleaning process:
- Original data: 541 valid rows (plus header)
- Data cleaning steps:
- Removed empty rows
- Standardized column names
- Recalculated derived columns (BMI, FSH/LH ratio, Waist:Hip ratio)
- Standardized binary columns (only actual Y/N columns)
- Converted data types appropriately
- Filled missing numerical values with medians

In [26]:
import pandas as pd
import numpy as np
import os

# Load the data
data_path = '../data/raw/PCOS_data_without_infertility.csv'
df = pd.read_csv(data_path)

# Drop empty rows (rows where all values are NaN or empty)
df = df.dropna(how='all')

print(f"Data shape after loading and dropping empty rows: {df.shape}")

# Display first few rows
df.head()

Data shape after loading and dropping empty rows: (541, 45)


,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm),Unnamed: 44
0,1.0,1.0,0.0,28.0,44.6,152.0,19.3,15.0,78.0,22.0,...,1.0,0.0,110.0,80.0,3.0,3.0,18.0,18.0,8.5,NaN
1,2.0,2.0,0.0,36.0,65.0,161.5,#NAME?,15.0,74.0,20.0,...,0.0,0.0,120.0,70.0,3.0,5.0,15.0,14.0,3.7,NaN
2,3.0,3.0,1.0,33.0,68.8,165.0,#NAME?,11.0,72.0,18.0,...,1.0,0.0,120.0,80.0,13.0,15.0,18.0,20.0,10.0,NaN
3,4.0,4.0,0.0,37.0,65.0,148.0,#NAME?,13.0,72.0,20.0,...,0.0,0.0,120.0,70.0,2.0,2.0,15.0,14.0,7.5,NaN
4,5.0,5.0,0.0,25.0,52.0,161.0,#NAME?,11.0,72.0,18.0,...,0.0,0.0,120.0,80.0,3.0,4.0,16.0,14.0,7.0,NaN


In [27]:
# Clean column names
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('/', '_').str.replace('.', '').str.lower()

# Manual fixes for specific columns
df.rename(columns={
    'cycler_i': 'cycle_r_i',
    'pregnanty_n': 'pregnant_y_n',
    'weight_gainy_n': 'weight_gain_y_n',
    'hair_growthy_n': 'hair_growth_y_n',
    'hair_lossy_n': 'hair_loss_y_n',
    'pimplesy_n': 'pimples_y_n',
    'fast_food_y_n': 'fast_food_y_n',
    'regexercisey_n': 'reg_exercise_y_n',
    'bp__systolic_mmhg': 'bp_systolic_mmhg',
    'bp__diastolic_mmhg': 'bp_diastolic_mmhg',
    'fshmiu_ml': 'fsh_miu_ml',
    'lhmiu_ml': 'lh_miu_ml',
    'i___beta-hcgmiu_ml': 'i_beta_hcg_miu_ml',
    'ii____beta-hcgmiu_ml': 'ii_beta_hcg_miu_ml',
    'waist:hip_ratio': 'waist_hip_ratio'
}, inplace=True)

# Drop unnamed columns
df = df.loc[:, ~df.columns.str.contains('unnamed')]

# Display cleaned column names
df.columns

Index(['sl_no', 'patient_file_no', 'pcos_y_n', 'age_yrs', 'weight_kg',
       'heightcm', 'bmi', 'blood_group', 'pulse_ratebpm', 'rr_breaths_min',
       'hbg_dl', 'cycle_r_i', 'cycle_lengthdays', 'marraige_status_yrs',
       'pregnant_y_n', 'no_of_aborptions', 'i_beta_hcg_miu_ml',
       'ii_beta_hcg_miu_ml', 'fsh_miu_ml', 'lh_miu_ml', 'fsh_lh', 'hipinch',
       'waistinch', 'waist_hip_ratio', 'tsh_miu_l', 'amhng_ml', 'prlng_ml',
       'vit_d3_ng_ml', 'prgng_ml', 'rbsmg_dl', 'weight_gain_y_n',
       'hair_growth_y_n', 'skin_darkening_y_n', 'hair_loss_y_n', 'pimples_y_n',
       'fast_food_y_n', 'reg_exercise_y_n', 'bp_systolic_mmhg',
       'bp_diastolic_mmhg', 'follicle_no_l', 'follicle_no_r',
       'avg_f_size_l_mm', 'avg_f_size_r_mm', 'endometrium_mm'],
      dtype='str')

In [28]:
# Recalculate derived columns
# BMI = Weight / (Height/100)^2
df['bmi'] = df['weight_kg'] / (df['heightcm'] / 100) ** 2

# FSH/LH ratio
df['fsh_lh'] = df['fsh_miu_ml'] / df['lh_miu_ml']

# Waist:Hip Ratio
df['waist_hip_ratio'] = df['waistinch'] / df['hipinch']

# Display info
df[['bmi', 'fsh_lh', 'waist_hip_ratio']].head()

,bmi,fsh_lh,waist_hip_ratio
0,19.304017,2.160326,0.833333
1,24.921163,6.174312,0.842105
2,25.270891,6.295455,0.900000
3,29.674945,3.415254,0.857143
4,20.060954,4.422222,0.810811


In [29]:
# Check for missing values
df.isnull().sum()

# Standardize binary columns (Y/N to 0/1)
binary_cols = ['pcos_y_n', 'pregnant_y_n', 'weight_gain_y_n', 'hair_growth_y_n', 
               'skin_darkening_y_n', 'hair_loss_y_n', 'pimples_y_n', 'fast_food_y_n', 'reg_exercise_y_n']

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map({'Y': 1, 'N': 0, 1: 1, 0: 0}).fillna(0).astype(int)

# Ensure numerical columns are float/int
numerical_cols = ['age_yrs', 'weight_kg', 'heightcm', 'pulse_ratebpm', 'rr_breaths_min', 
                  'hbg_dl', 'cycle_lengthdays', 'marraige_status_yrs', 'no._of_aborptions', 
                  'i_beta-hcgmiu_ml', 'ii_beta-hcgmiu_ml', 'fshm_iu_ml', 'lhm_iu_ml', 
                  'hipinch', 'waistinch', 'tsh_miu_l', 'amhng_ml', 'prlng_ml', 
                  'vit_d3_ng_ml', 'prgng_ml', 'rbsmg_dl', 'bp_systolic_mmhg', 
                  'bp_diastolic_mmhg', 'follicle_no._l', 'follicle_no._r', 
                  'avg._f_size_l_mm', 'avg._f_size_r_mm', 'endometrium_mm']

for col in numerical_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill any remaining NaN with median for numerical columns
for col in numerical_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Column types after cleaning
df.dtypes

/var/folders/xy/9fy3br45203b38cd8vm7gdgw0000gn/T/ipykernel_32406/1122491177.py:28: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(df[col].median(), inplace=True)
/var/folders/xy/9fy3br45203b38cd8vm7gdgw0000gn/T/ipykernel_32406/1122491177.py:28: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Ser

sl_no                  float64
patient_file_no        float64
pcos_y_n                 int64
age_yrs                float64
weight_kg              float64
heightcm               float64
bmi                    float64
blood_group            float64
pulse_ratebpm          float64
rr_breaths_min         float64
hbg_dl                 float64
cycle_r_i              float64
cycle_lengthdays       float64
marraige_status_yrs    float64
pregnant_y_n             int64
no_of_aborptions       float64
i_beta_hcg_miu_ml      float64
ii_beta_hcg_miu_ml         str
fsh_miu_ml             float64
lh_miu_ml              float64
fsh_lh                 float64
hipinch                float64
waistinch              float64
waist_hip_ratio        float64
tsh_miu_l              float64
amhng_ml               float64
prlng_ml               float64
vit_d3_ng_ml           float64
prgng_ml               float64
rbsmg_dl               float64
weight_gain_y_n          int64
hair_growth_y_n          int64
skin_dar

In [30]:
# Save the cleaned data
output_path = '../data/processed/cleaned_data.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)

print(f"Cleaned data saved to {output_path}")
df.info()

Cleaned data saved to ../data/processed/cleaned_data.csv
<class 'pandas.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 44 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   sl_no                541 non-null    float64
 1   patient_file_no      541 non-null    float64
 2   pcos_y_n             541 non-null    int64  
 3   age_yrs              541 non-null    float64
 4   weight_kg            541 non-null    float64
 5   heightcm             541 non-null    float64
 6   bmi                  541 non-null    float64
 7   blood_group          541 non-null    float64
 8   pulse_ratebpm        541 non-null    float64
 9   rr_breaths_min       541 non-null    float64
 10  hbg_dl               541 non-null    float64
 11  cycle_r_i            541 non-null    float64
 12  cycle_lengthdays     541 non-null    float64
 13  marraige_status_yrs  540 non-null    float64
 14  pregnant_y_n         541 non-null    int64  